# Notebook 3: FP Projection Model

**Goal:** Train a model that predicts DraftKings fantasy points.

**Input:** `yakos_features.parquet`

**Output:** `models/yakos_fp_model.pkl`, evaluation metrics, feature importance chart

## Evaluation Targets
- MAE ≤ 8 FP (current system ~21 FP — broken)
- Bias ±1 FP (current +20.8 — fundamentally wrong)
- Hit rate (±10 FP) ≥ 50%

## Model Progression
1. **V1:** Ridge Regression — fast baseline
2. **V2:** LightGBM — captures nonlinear interactions
3. **V3:** Ensemble — 40% YakOS + 30% Tank01 + 30% RG

In [ ]:
# !pip install scikit-learn lightgbm joblib pyarrow -q

import os
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

INPUT_PATH  = 'yakos_features.parquet'
MODEL_DIR   = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

df = pd.read_parquet(INPUT_PATH)
df['game_date'] = pd.to_datetime(df['game_date'])
print(f'Features shape: {df.shape}')
print(f'Date range: {df["game_date"].min()} to {df["game_date"].max()}')

## Train/Test Split (No Leakage)

In [ ]:
CUTOFF_DATE = pd.Timestamp('2026-02-15')

FEATURE_COLS = [
    'salary', 'tank01_proj', 'rg_proj',
    'rolling_fp_5', 'rolling_fp_10', 'rolling_fp_20',
    'rolling_min_5', 'rolling_min_10', 'min_std_10',
    'dvp', 'vegas_total', 'vegas_spread',
    'home', 'b2b', 'days_rest',
    'rolling_usage_10', 'trend', 'value_score',
]
TARGET = 'dk_fp_actual'

available_features = [c for c in FEATURE_COLS if c in df.columns]
print(f'Using {len(available_features)} features: {available_features}')

train_df = df[df['game_date'] < CUTOFF_DATE].copy()
val_df   = df[df['game_date'] >= CUTOFF_DATE].copy()

# Drop rows where target is missing
train_df = train_df.dropna(subset=[TARGET])
val_df   = val_df.dropna(subset=[TARGET])

X_train = train_df[available_features]
y_train = train_df[TARGET]
X_val   = val_df[available_features]
y_val   = val_df[TARGET]

print(f'Train: {len(X_train)} rows | Val: {len(X_val)} rows')

## V1: Ridge Regression Baseline

In [ ]:
ridge_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   Ridge(alpha=10.0)),
])

ridge_pipeline.fit(X_train, y_train)
ridge_preds = ridge_pipeline.predict(X_val)

mae_ridge  = mean_absolute_error(y_val, ridge_preds)
rmse_ridge = np.sqrt(mean_squared_error(y_val, ridge_preds))
bias_ridge = (ridge_preds - y_val).mean()
hit_rate   = ((np.abs(ridge_preds - y_val) <= 10).mean() * 100)

print('=== Ridge Regression (V1) ===')
print(f'  MAE:      {mae_ridge:.2f} FP   (target ≤ 8)')
print(f'  RMSE:     {rmse_ridge:.2f} FP')
print(f'  Bias:     {bias_ridge:+.2f} FP  (target ±1)')
print(f'  Hit Rate: {hit_rate:.1f}%       (target ≥ 50%)')

## V2: LightGBM (Nonlinear)

In [ ]:
try:
    import lightgbm as lgb

    # Impute before LightGBM (it can handle NaN natively but explicit is safer)
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy='median')
    X_train_imp = imp.fit_transform(X_train)
    X_val_imp   = imp.transform(X_val)

    lgb_model = lgb.LGBMRegressor(
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1,
    )
    lgb_model.fit(
        X_train_imp, y_train,
        eval_set=[(X_val_imp, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)],
    )
    lgb_preds = lgb_model.predict(X_val_imp)

    mae_lgb  = mean_absolute_error(y_val, lgb_preds)
    rmse_lgb = np.sqrt(mean_squared_error(y_val, lgb_preds))
    bias_lgb = (lgb_preds - y_val).mean()
    hit_lgb  = ((np.abs(lgb_preds - y_val) <= 10).mean() * 100)

    print('=== LightGBM (V2) ===')
    print(f'  MAE:      {mae_lgb:.2f} FP   (target ≤ 8)')
    print(f'  RMSE:     {rmse_lgb:.2f} FP')
    print(f'  Bias:     {bias_lgb:+.2f} FP  (target ±1)')
    print(f'  Hit Rate: {hit_lgb:.1f}%       (target ≥ 50%)')

    BEST_MODEL = 'lgb'
except ImportError:
    print('LightGBM not installed. Using Ridge as best model.')
    BEST_MODEL = 'ridge'

## V3: Ensemble Blend (40% YakOS + 30% Tank01 + 30% RG)

In [ ]:
def ensemble_blend(yakos_preds, tank01_preds, rg_preds,
                   w_yakos=0.40, w_tank01=0.30, w_rg=0.30):
    """Blend three projection sources with redistribution for missing values."""
    result = np.zeros(len(yakos_preds))
    for i in range(len(yakos_preds)):
        vals, weights = [], []
        for v, w in [(yakos_preds[i], w_yakos), (tank01_preds[i], w_tank01), (rg_preds[i], w_rg)]:
            if v is not None and not np.isnan(float(v)):
                vals.append(float(v))
                weights.append(w)
        total_w = sum(weights)
        result[i] = sum(v*w for v,w in zip(vals,weights)) / total_w if total_w > 0 else 0.0
    return result


# Build ensemble using best YakOS model + consensus
yakos_preds_v = lgb_preds if BEST_MODEL == 'lgb' else ridge_preds
tank01_preds_v = val_df['tank01_proj'].fillna(0).values
rg_preds_v     = val_df['rg_proj'].fillna(np.nan).values

ensemble_preds = ensemble_blend(yakos_preds_v, tank01_preds_v, rg_preds_v)

mae_ens  = mean_absolute_error(y_val, ensemble_preds)
bias_ens = (ensemble_preds - y_val).mean()
hit_ens  = ((np.abs(ensemble_preds - y_val) <= 10).mean() * 100)

print('=== V3 Ensemble (40% YakOS + 30% Tank01 + 30% RG) ===')
print(f'  MAE:      {mae_ens:.2f} FP')
print(f'  Bias:     {bias_ens:+.2f} FP')
print(f'  Hit Rate: {hit_ens:.1f}%')

## Feature Importance

In [ ]:
try:
    import matplotlib.pyplot as plt

    if BEST_MODEL == 'lgb':
        importance = pd.Series(
            lgb_model.feature_importances_,
            index=available_features
        ).sort_values(ascending=True)
    else:
        coef = ridge_pipeline.named_steps['model'].coef_
        importance = pd.Series(
            np.abs(coef), index=available_features
        ).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(8, max(4, len(importance) * 0.4)))
    importance.plot.barh(ax=ax, color='steelblue')
    ax.set_title('Feature Importance — YakOS FP Model')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig('yakos_fp_feature_importance.png', dpi=150)
    plt.show()
    print('Feature importance chart saved.')
except Exception as e:
    print(f'Chart skipped: {e}')

## Save Best Model

In [ ]:
# Save the best-performing single model (ensemble is assembled at inference time)
model_to_save = lgb_model if BEST_MODEL == 'lgb' else ridge_pipeline
model_path = MODEL_DIR / 'yakos_fp_model.pkl'
joblib.dump(model_to_save, model_path)
print(f'Model saved: {model_path}')

# Save imputer separately (needed if LightGBM was chosen)
if BEST_MODEL == 'lgb':
    joblib.dump(imp, MODEL_DIR / 'yakos_fp_imputer.pkl')

# Save feature list used by the model
import json
with open(MODEL_DIR / 'yakos_fp_features.json', 'w') as f:
    json.dump(available_features, f)

metrics = {
    'model_type': BEST_MODEL,
    'mae': float(mae_lgb if BEST_MODEL == 'lgb' else mae_ridge),
    'bias': float(bias_lgb if BEST_MODEL == 'lgb' else bias_ridge),
    'hit_rate_10fp': float(hit_lgb if BEST_MODEL == 'lgb' else hit_rate),
    'ensemble_mae': float(mae_ens),
}
with open(MODEL_DIR / 'yakos_fp_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved.')
print(metrics)